# 01 — Exploratory Data Analysis

First look at the raw HR extract in `data/raw/`, before any cleaning. Eight specific data-quality issues are confirmed below (grouped into 6 categories); each is fixed with a stated rule in `02_data_cleaning.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('data/raw')
employees = pd.read_csv(RAW / 'employees.csv')
compensation = pd.read_csv(RAW / 'compensation.csv')
performance = pd.read_csv(RAW / 'performance_reviews.csv')
exit_surveys = pd.read_csv(RAW / 'exit_surveys.csv')
departments = pd.read_csv(RAW / 'departments.csv')

print('employees:', employees.shape)
print('compensation:', compensation.shape)
print('performance_reviews:', performance.shape)
print('exit_surveys:', exit_surveys.shape)

employees: (3347, 14)
compensation: (3347, 4)
performance_reviews: (8391, 5)
exit_surveys: (991, 7)


In [2]:
employees.head()

,employee_id,department,job_role,job_level,gender,age,education_level,marital_status,hire_date,termination_date,termination_reason,active_status,distance_from_home_miles,remote_flag
0,EMP-11829,Operations,Facilities Coordinator,1,Male,37,Bachelor's,Divorced,2022-08-25,NaN,NaN,1,1.5,False
1,EMP-12626,Finance,Financial Analyst,1,Female,28,Master's,Single,2022-11-20,NaN,NaN,1,42.5,False
2,EMP-11551,Customer Success,Customer Success Rep,1,Female,35,Bachelor's,Single,2022-06-10,NaN,NaN,1,7.6,False
3,EMP-11236,Customer Success,CS Team Lead,3,Female,43,Bachelor's,Single,2021-05-12,2024-03-14,Better Opportunity,0,0.5,False
4,EMP-12973,Engineering,Engineering Manager,3,Male,32,Master's,Married,2024-10-06,2025-05-18,Better Opportunity,0,1.0,True


## Issue 1 — Duplicate employee records

In [3]:
dupe_count = employees['employee_id'].duplicated().sum()
print(f'{dupe_count} duplicate employee_id rows (re-ingested during an HRIS migration).')

47 duplicate employee_id rows (re-ingested during an HRIS migration).


## Issue 2 — Inconsistent department labels

In [4]:
print('Unique department label variants in raw data:', employees["department"].nunique())
employees['department'].value_counts().head(12)

Unique department label variants in raw data: 27


department
Engineering         1083
Sales                618
Customer Success     390
Marketing            320
Finance              276
Operations           185
HR                   181
 Engineering          38
ENGINEERING           35
sales                 25
engineering           25
SALES                 21
Name: count, dtype: int64

## Issue 3 — Status-sync gap: `active_status = 0` but no `termination_date`

In [5]:
gap = (employees['active_status']==0) & (employees['termination_date'].isna())
print(f'{gap.sum()} employees marked departed with no termination_date recorded.')

125 employees marked departed with no termination_date recorded.


## Issue 4 — `hire_date` after `termination_date` (impossible)

In [6]:
emp_dates = employees.copy()
emp_dates['hire_date'] = pd.to_datetime(emp_dates['hire_date'], errors='coerce')
emp_dates['termination_date'] = pd.to_datetime(emp_dates['termination_date'], errors='coerce')
swapped = emp_dates['termination_date'].notna() & (emp_dates['hire_date'] > emp_dates['termination_date'])
print(f'{swapped.sum()} rows have hire_date AFTER termination_date.')
emp_dates[swapped][['employee_id','hire_date','termination_date']].head()

33 rows have hire_date AFTER termination_date.


,employee_id,hire_date,termination_date
65,EMP-11242,2025-12-23,2022-02-12
159,EMP-12357,2023-07-31,2022-09-03
221,EMP-10776,2023-03-20,2023-02-07
254,EMP-10144,2025-04-20,2022-07-28
419,EMP-10443,2024-12-26,2021-04-13


## Issue 5 — Zero/negative `distance_from_home_miles`

In [7]:
bad_dist = (employees['distance_from_home_miles'] <= 0).sum()
print(f'{bad_dist} rows have distance_from_home_miles <= 0 (GPS/geocoding glitch).')

58 rows have distance_from_home_miles <= 0 (GPS/geocoding glitch).


## Issue 6 — Zero/negative `base_salary`

In [8]:
bad_sal = (compensation['base_salary'] <= 0).sum()
print(f'{bad_sal} compensation rows have base_salary <= 0 (payroll sync glitch).')

61 compensation rows have base_salary <= 0 (payroll sync glitch).


## Issue 7 — Performance scores outside the valid 1-5 range

In [9]:
bad_perf = (~performance['performance_score'].between(1,5)).sum()
print(f'{bad_perf} performance review rows have a score outside 1-5.')
performance[~performance['performance_score'].between(1,5)].head()

41 performance review rows have a score outside 1-5.


,review_id,employee_id,review_date,performance_score,promotion_flag
165,166,EMP-10051,2023-12-28,6,False
406,407,EMP-10805,2025-07-06,7,False
444,445,EMP-12303,2023-03-03,7,False
567,568,EMP-11697,2024-06-03,0,False
609,610,EMP-10670,2021-06-20,0,False


## Issue 8 — Orphaned exit-survey rows (no matching employee)

In [10]:
orphans = ~exit_surveys['employee_id'].isin(employees['employee_id'])
print(f'{orphans.sum()} exit survey rows reference an employee_id that does not exist \n'
      f'in the employees table — leftovers from a prior HRIS migration.')

24 exit survey rows reference an employee_id that does not exist 
in the employees table — leftovers from a prior HRIS migration.


## Summary of confirmed issues

| # | Issue | Fix applied in 02_data_cleaning |
|---|---|---|
| 1 | Duplicate employee_id rows | Dropped, kept first |
| 2 | Inconsistent department labels (27 raw variants) | Standardized to 7 canonical values |
| 3 | Status-sync gap (departed, no term date) | Flagged `data_quality_flag`, not imputed |
| 4 | Swapped hire/termination dates | Corrected (fields swapped back) |
| 5 | Invalid commute distance | Flagged `is_valid_distance = False` |
| 6 | Invalid base salary | Flagged `is_valid_salary = False` |
| 7 | Invalid performance score | Flagged `is_valid_score = False` |
| 8 | Orphaned exit surveys | Dropped (no employee to attribute them to) |